# Kaggle Camera-Ready Pipeline

Run this notebook on Kaggle (GPU + Internet enabled) to reproduce the paper artefacts from the anonymised archive. Each cell maps to a review checklist item so that reviewers can execute them sequentially without modification.

## Execution Overview

1. **Unpack `artifact_anonymous.zip`** – extracts the anonymised repository into `/kaggle/working/artifact`.
2. **Install the package** – installs the project in editable mode so entry points resolve.
3. **Capture environment versions** – records Python, platform, package, and Torch runtime details.
4. **Run the camera-ready pipeline** – launches `scripts/reproduce_all.sh` with its default Kaggle paths.
5. **Preview paper outputs** – lists `/kaggle/working/paper_outputs` and prints `paper_status.txt`.

> ℹ️ Upload `artifact_anonymous.zip` as a Kaggle Dataset input before starting the notebook. The unzip step automatically discovers the dataset regardless of the dataset slug.

In [ ]:
import pathlib
import subprocess

INPUT_ROOT = pathlib.Path("/kaggle/input")
zip_candidates = sorted(INPUT_ROOT.glob("**/artifact_anonymous.zip"))
if not zip_candidates:
    raise FileNotFoundError(
        "Upload artifact_anonymous.zip as a Kaggle Dataset input before running this cell."
    )

ARTIFACT_ZIP = zip_candidates[0]
ARTIFACT_DIR = pathlib.Path("/kaggle/working/artifact")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Using archive: {ARTIFACT_ZIP}")
subprocess.run(["unzip", "-q", str(ARTIFACT_ZIP), "-d", str(ARTIFACT_DIR)], check=True)
print(f"Extracted into: {ARTIFACT_DIR}")


In [ ]:
import subprocess
import sys
from pathlib import Path

ARTIFACT_DIR = Path("/kaggle/working/artifact")
if not ARTIFACT_DIR.exists():
    raise FileNotFoundError("Run the unzip cell first to populate /kaggle/working/artifact.")

subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(ARTIFACT_DIR)], check=True)


In [ ]:
import importlib.metadata
import platform
import sys

print(f"python_version={sys.version}")
print(f"platform={platform.platform()}")

try:
    pkg_version = importlib.metadata.version("leadlag-signature-rl")
except importlib.metadata.PackageNotFoundError as err:  # pragma: no cover - Kaggle safety net
    raise RuntimeError(
        "leadlag-signature-rl is not installed. Run the install cell first."
    ) from err
else:
    print(f"leadlag-signature-rl={pkg_version}")

try:
    import torch
except Exception as exc:  # pragma: no cover - diagnostics only
    print(f"torch import failed: {exc}")
else:
    print(f"torch_version={torch.__version__}")
    print(f"cuda_available={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"cuda_device={torch.cuda.get_device_name(0)}")


In [ ]:
import subprocess
from pathlib import Path

script_path = Path("/kaggle/working/artifact/scripts/reproduce_all.sh")
if not script_path.exists():
    raise FileNotFoundError(
        "Expected scripts/reproduce_all.sh inside the unpacked artifact."
    )

subprocess.run(["bash", str(script_path)], check=True)


In [ ]:
from pathlib import Path

paper_root = Path("/kaggle/working/paper_outputs")
if not paper_root.exists():
    raise FileNotFoundError(
        "Paper outputs not found. Ensure the reproduce script completed successfully."
    )

print("paper_outputs contents:")
for path in sorted(paper_root.iterdir()):
    if path.is_dir():
        print(f"- [dir] {path.name}")
    else:
        print(f"- {path.name}")

status_file = paper_root / "paper_status.txt"
if status_file.exists():
    print("
Summary:")
    print(status_file.read_text())
else:
    print("
[pending] paper_status.txt not found; check pipeline logs under results/.")
